In [ ]:
from pyspark.sql import SparkSession, column
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, IntegerType, DoubleType, StructType, StructField, DateType, DecimalType, LongType, BooleanType, TimestampType, FloatType, NumericType

In [ ]:
# Sessão Spark conectada ao MinIO (s3a)
spark = (
    SparkSession.builder.appName("silver.py")
    .master("spark://spark-master:7077")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minio")
    .config("spark.hadoop.fs.s3a.secret.key", "minio123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.fast.upload", "true")
    .config("spark.driver.extraClassPath", "/home/jovyan/extra-jars/*")
    .config("spark.executor.extraClassPath", "/opt/spark/extra-jars/*")
    .getOrCreate()
)

In [ ]:
spark.getActiveSession()

In [ ]:
# Leitura da camada bronze
df_original = spark.read.csv("s3a://bronze/kc_house_data.csv", header=True)

In [ ]:
df_original.printSchema()

In [ ]:
df_original.show(10)

In [ ]:
# Base de trabalho e remoção de espaços em branco
df_columns = df_original.columns
df_sem_espacos = df_original

# Remoção de espaços -> ""
for c in df_columns:
    df_sem_espacos = df_sem_espacos.withColumn(c, F.regexp_replace(c, r"\s+", ""))

# Quantidade de registros com string vazia ("")
df_sem_espacos.select([
    F.sum((F.col(c) == "").cast("int")).alias(c)
    for c in df_columns
]).show()

# Quantidade de registros NULL / None
df_sem_espacos.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_columns
]).show()

# Validação de duplicadas
df_sem_espacos.groupBy(df_columns).count().filter(F.col("count") > 1).count()

In [ ]:
# Tratamento de price: padrões (ponto / vírgula / inválido)
df_sem_espacos.select("price").describe().show()

df_preco_padroes = df_sem_espacos.select(
    "price",
    F.when(F.col("price").rlike(r"^\d+(\.\d+)?$"), "ponto")
     .when(F.col("price").rlike(r"^\d+(\,\d+)?$"), "virgula")
     .otherwise("invalido")
     .alias("padrao")
)

# Separar padrões
df_preco_padroes.groupBy(F.col("padrao")).count().show()

# Ver tipos de inválidos
df_preco_padroes.filter(F.col("padrao") == "invalido").select("price").show()

In [ ]:
# Validação das colunas numéricas (apenas dígitos, com ponto e vírgula)
df_numerico = df_sem_espacos
colunas_selecionadas = [c for c in df_original.columns if c not in ["id", "date", "waterfront", "view", "price", "long"]]

df_numerico.select([
    F.col(c).rlike(r"^\d+([.,]\d+)?$").alias(c) for c in colunas_selecionadas
]).filter(F.col(c) == False).show()

# Validado que as colunas_selecionadas possuem apenas simbologia numérica
# (com ponto e vírgula), sem letras nem caracteres especiais.

In [ ]:
# Validação de date via máscara: dígito -> 9, letra -> A
df_colunas_restantes = df_sem_espacos

df_colunas_restantes.select(
    F.regexp_replace(
        F.regexp_replace(F.col("date"), r"[0-9]", "9"),
        r"[A-Za-z]",
        "A"
    ).alias("padrao")
).groupBy(F.col("padrao")).count().orderBy(F.desc("count")).show()

# Somente um padrão encontrado (yyyyMMddThhmmss)

In [ ]:
# Validação de waterfront
df_colunas_restantes.groupBy(F.col("waterfront")).count().show()
# Somente 1 ou 0 (considerar true / false)

df_para_cast = df_sem_espacos
df_para_cast = df_para_cast.withColumn("waterfront", F.when(F.col("waterfront") == 1, True).otherwise(False))

# Validação do valor (cast ainda necessário)
df_para_cast.groupBy(F.col("waterfront")).count().show()

In [ ]:
# Validação de view (padrões)
df_para_cast.groupBy(F.col("view")).count().show()

In [ ]:
# Validação de long: aceita "-" no início, ponto e vírgula; restante apenas dígitos
df_para_cast.select(
    F.col("long").rlike(r"^-?\d+([.,]\d+)?$").alias("long_rlike")
).filter(F.col("long_rlike") != True).show()

In [ ]:
# Validação de id (apenas dígitos)
df_para_cast.select(
    F.col("id").rlike(r"^\d+$").alias("id")
).filter(F.col("id") != True).show()

In [ ]:
# Conferência: amostra e valores distintos de floors
df_para_cast.show(1)
df_para_cast.select(F.col("floors")).distinct().show()

In [ ]:
# Cast final dos tipos (camada silver) — a partir de df_sem_espacos
df_cast = (
    df_sem_espacos
    .withColumn("id", F.col("id").cast(LongType()))
    .withColumn("date", F.to_timestamp("date", "yyyyMMdd'T'HHmmss").cast(TimestampType()))
    .withColumn("price", F.col("price").cast(DoubleType()))
    .withColumn("bathrooms", F.floor(F.col("bathrooms").cast(DoubleType())))
    .withColumn("bedrooms", F.floor(F.col("bedrooms").cast(IntegerType())))
    .withColumn("sqft_living", F.col("sqft_living").cast(DoubleType()))
    .withColumn("sqft_lot", F.col("sqft_lot").cast(DoubleType()))
    .withColumn("floors", F.col("floors").cast(DoubleType()))
    .withColumn("waterfront", F.col("waterfront").cast(BooleanType()))
    .withColumn("view", F.floor(F.col("view").cast(IntegerType())))
    .withColumn("condition", F.col("condition").cast(DoubleType()))
    .withColumn("grade", F.col("grade").cast(DoubleType()))
    .withColumn("sqft_above", F.col("sqft_above").cast(DoubleType()))
    .withColumn("sqft_basement", F.col("sqft_basement").cast(DoubleType()))
    .withColumn("yr_built", F.col("yr_built").cast(IntegerType()))
    .withColumn("yr_renovated", F.col("yr_renovated").cast(IntegerType()))
    .withColumn("zipcode", F.col("zipcode").cast(StringType()))
    .withColumn("lat", F.col("lat").cast(DoubleType()))
    .withColumn("long", F.col("long").cast(DoubleType()))
    .withColumn("sqft_living15", F.col("sqft_living15").cast(DoubleType()))
    .withColumn("sqft_lot15", F.col("sqft_lot15").cast(DoubleType()))
)

df_cast.show(5)
df_cast.printSchema()

In [ ]:
# Validações — Regra de negócio
# sqft_living     Área útil habitável da casa.
# sqft_lot        Área total do terreno.
# sqft_above      Área habitável acima do nível do solo.
# sqft_basement   Área do porão (subsolo).
# sqft_living15   Média da área habitável das 15 casas vizinhas mais próximas.
# sqft_lot15      Média da área dos terrenos das 15 casas vizinhas mais próximas.

In [ ]:
# Floors com parte decimal — inspecionar contra sqft_basement
df_cast.select("id", "floors", "sqft_basement").where(F.col("floors") % 1 != 0).show()
df_cast.select("id", "floors", "sqft_basement").where(F.col("floors") % 1 != 0).count()

In [ ]:
# Regra: se floors tem decimal e não há porão (sqft_basement <= 0), arredonda para baixo;
# caso contrário, mantém o decimal.
df_cast = df_cast.withColumn(
    "floors",
    F.when(
        (F.col("floors") % 1 != 0) & (F.col("sqft_basement") <= 0),
        F.floor(F.col("floors"))
    ).otherwise(F.col("floors"))
)

# Validar aplicação (comparar count com a célula anterior)
df_cast.select(
    "id", "floors", "sqft_basement"
).where(
    (F.col("floors") % 1 != 0) & (F.col("sqft_basement") <= 0)
).show()